In [1]:
from transformer_lens import HookedTransformer

/Users/danielkrawciw/Documents/llms/logit/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model: HookedTransformer = HookedTransformer.from_pretrained("gpt2-small")

logits, cache = model.run_with_cache("The sky is blue and the grass is green.")
print(logits.shape)

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model gpt2-small into HookedTransformer
torch.Size([1, 11, 50257])


In [57]:
prompt = "President Obama's last name is"
tokens = model.to_tokens(prompt)
logits, cache = model.run_with_cache(tokens)

In [58]:
for layer in range(model.cfg.n_layers):
    resid = cache[f"resid_post", layer]

    logits = model.unembed(resid)
    probs = logits.softmax(dim=-1)

    top_tokens = probs[0, -1].topk(5)
    print(layer, model.to_str_tokens(top_tokens.indices))

0 [' not', ' now', ' still', ' unlikely', ' also']
1 [' now', ' not', ' unlikely', ' currently', ' still']
2 [' now', ' not', ' currently', ' unlikely', ' still']
3 [' now', ' currently', ' not', ' still', ' likely']
4 [' now', ' not', ' currently', ' still', ' clearly']
5 [' synonymous', ' now', ' not', ' currently', ' still']
6 [' synonymous', ' now', ' not', ' spelled', ' listed']
7 [' synonymous', ' spelled', ' pronounced', ' listed', ' now']
8 [' synonymous', ' spelled', ' pronounced', ' listed', ' redacted']
9 [' spelled', ' pronounced', ' synonymous', ' redacted', ' listed']
10 [' spelled', ' pronounced', ' synonymous', ' redacted', ' listed']
11 [' spelled', ' pronounced', ' synonymous', ' redacted', ' Barack']


# Tuned Lens

In [ ]:
# We want to create a tuned lens now
for layer in range(model.cfg.n_layers):
    resid = cache[f"resid_post", layer]

    logits = model.unembed(resid)
    probs = logits.softmax(dim=-1)

    print(f"Layer {layer} top tokens:")
    top_tokens = probs[0, -1].topk(5)
    print(model.to_str_tokens(top_tokens.indices))

Layer 0 top tokens:
[' not', ' now', ' still', ' unlikely', ' also']
Layer 1 top tokens:
[' now', ' not', ' unlikely', ' currently', ' still']
Layer 2 top tokens:
[' now', ' not', ' currently', ' unlikely', ' still']
Layer 3 top tokens:
[' now', ' currently', ' not', ' still', ' likely']
Layer 4 top tokens:
[' now', ' not', ' currently', ' still', ' clearly']
Layer 5 top tokens:
[' synonymous', ' now', ' not', ' currently', ' still']
Layer 6 top tokens:
[' synonymous', ' now', ' not', ' spelled', ' listed']
Layer 7 top tokens:
[' synonymous', ' spelled', ' pronounced', ' listed', ' now']
Layer 8 top tokens:
[' synonymous', ' spelled', ' pronounced', ' listed', ' redacted']
Layer 9 top tokens:
[' spelled', ' pronounced', ' synonymous', ' redacted', ' listed']
Layer 10 top tokens:
[' spelled', ' pronounced', ' synonymous', ' redacted', ' listed']
Layer 11 top tokens:
[' spelled', ' pronounced', ' synonymous', ' redacted', ' Barack']


In [ ]:
import torch
from tuned_lens import TunedLens
from tuned_lens.nn.lenses import TunedLens
import matplotlib.pyplot as plt
import numpy as np

# ── 1. Load model ────────────────────────────────────────────────────────────
model = HookedTransformer.from_pretrained("gpt2")
model.eval()

# ── 2. Load the pre-trained tuned lens for GPT-2 small ───────────────────────
# Pre-trained lenses are hosted on HuggingFace: "AlignmentResearch/tuned-lens"
tuned_lens = TunedLens.from_unembed_matrix(
    unembed=model.unembed,                  # the unembedding matrix W_U
    num_layers=model.cfg.n_layers,          # 12 for GPT-2 small
)

# Or load directly from HuggingFace Hub (recommended):
tuned_lens = TunedLens.from_pretrained("gpt2", map_location="cpu")

# ── 3. Tokenize input ─────────────────────────────────────────────────────────
prompt = "The Eiffel Tower is located in"
tokens = model.to_tokens(prompt)            # shape: [1, seq_len]
token_strs = model.to_str_tokens(prompt)

# ── 4. Run model and collect hidden states ────────────────────────────────────
with torch.no_grad():
    # cache contains all intermediate residual stream activations
    logits, cache = model.run_with_cache(tokens)

# ── 5. Decode each layer's hidden state with the tuned lens ───────────────────
n_layers = model.cfg.n_layers   # 12
results = []

with torch.no_grad():
    for layer in range(n_layers):
        # Get residual stream AFTER each transformer block
        resid = cache[f"blocks.{layer}.hook_resid_post"]  # [1, seq_len, d_model]

        # Project hidden state → vocab logits using the tuned lens translator
        layer_logits = tuned_lens(resid, layer)            # [1, seq_len, vocab_size]
        layer_probs  = torch.softmax(layer_logits, dim=-1)

        # Top-1 predicted token at final token position
        top_token_id  = layer_probs[0, -1].argmax()
        top_token_str = model.to_string(top_token_id)
        top_prob      = layer_probs[0, -1].max().item()

        results.append({
            "layer": layer,
            "top_token": top_token_str,
            "prob": top_prob,
        })
        print(f"Layer {layer:2d} | top token: {top_token_str!r:15s} | prob: {top_prob:.3f}")

# ── 6. Visualise prediction evolution across layers ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: probability of the final model's top-1 token at each layer
final_token = model.to_string(logits[0, -1].argmax())
print(f"\nFinal model prediction: {final_token!r}")

# Build probability matrix [n_layers × seq_len] for a heatmap
prob_matrix = np.zeros((n_layers, len(token_strs)))

with torch.no_grad():
    for layer in range(n_layers):
        resid = cache[f"blocks.{layer}.hook_resid_post"]
        lp    = torch.softmax(tuned_lens(resid, layer), dim=-1)
        # entropy as a proxy for confidence
        entropy = -(lp * lp.log().clamp(min=-1e9)).sum(-1)  # [1, seq_len]
        prob_matrix[layer] = entropy[0].cpu().numpy()

ax = axes[0]
im = ax.imshow(prob_matrix, aspect="auto", cmap="viridis_r")
ax.set_xticks(range(len(token_strs)))
ax.set_xticklabels(token_strs, rotation=45, ha="right")
ax.set_yticks(range(n_layers))
ax.set_yticklabels([f"L{i}" for i in range(n_layers)])
ax.set_title("Prediction Entropy per Layer & Token\n(darker = more confident)")
plt.colorbar(im, ax=ax, label="Entropy")

# Right: top-token probability at the last position across layers
ax2 = axes[1]
probs = [r["prob"] for r in results]
ax2.plot(range(n_layers), probs, marker="o")
ax2.set_xlabel("Layer")
ax2.set_ylabel("Top-token probability")
ax2.set_title(f'Confidence for final-position top token\n(prompt: "{prompt}")')
ax2.set_xticks(range(n_layers))
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("tuned_lens_inference.png", dpi=150)
plt.show()